# Prática 2 — O canal que respira

Esta prática refaz, com `pandas`, a caçada que a Seção~1.6 do capítulo descreve no Kibana.
Cada passo mostra primeiro a **consulta ingênua**, que cai na isca, e depois a **pergunta refinada**,
que responde à hipótese. No fim, as respostas são conferidas contra o gabarito que o gerador do
cenário recalcula por código.

**Hipótese.** Um implante que pede ordens consulta o mesmo nome com regularidade de relógio, a partir de poucas máquinas, e a exfiltração aparece na soma do que sai por destino.

## Carregar a evidência

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))
import pandas as pd

from eventos import conferir, ler_eventos, ler_gabarito

eventos = ler_eventos()
gabarito = ler_gabarito()
print(f"{len(eventos)} eventos, de {eventos['@timestamp'].min()} a {eventos['@timestamp'].max()}")
eventos["event.dataset"].value_counts()

49277 eventos, de 2026-03-09 00:00:07+00:00 a 2026-03-11 23:59:42+00:00


event.dataset
dns           22159
firewall       9655
fileserver     7340
edr            4696
vpn            2967
auth           2460
Name: count, dtype: int64

## Passo 1 — o domínio mais consultado, e por que ele engana

```
event.dataset:"dns"  →  contagem por dns.question.name
```

In [2]:
dns = eventos[eventos["event.dataset"] == "dns"]
dns["dns.question.name"].value_counts().head(5)

dns.question.name
update.aurora.com.br     8640
api.whatsapp.com         1109
www.gov.br               1090
www.youtube.com          1083
outlook.office365.com    1083
Name: count, dtype: int64

## Passo 2 — frequência com raridade

O domínio do ataque não é o mais consultado: é o consultado **por uma máquina só**.

In [3]:
por_dominio = dns.groupby("dns.question.name").agg(consultas=("host.name", "size"),
                                                  estacoes=("host.name", "nunique"))
suspeitos = por_dominio[por_dominio["estacoes"] == 1].sort_values("consultas", ascending=False)
dominio = suspeitos.index[0]
estacao = dns[dns["dns.question.name"] == dominio]["host.name"].iloc[0]
print(dominio, estacao)
suspeitos.head(5)

telemetria-aurora-cdn.net WKS-ENG-117


,consultas,estacoes
dns.question.name,,
telemetria-aurora-cdn.net,719,1
catalogo-antenas.net,3,1
cotacao-fibra.com,3,1
fornecedor-cabos.com.br,3,1


## Passo 3 — a regularidade

In [4]:
consultas = dns[dns["dns.question.name"] == dominio].sort_values("@timestamp")
intervalos = consultas["@timestamp"].diff().dt.total_seconds().dropna()
intervalo = int(intervalos.median())
print(f"mediana {intervalo} s, de {intervalos.min():.0f} a {intervalos.max():.0f} s, "
      f"em {len(consultas)} consultas")

mediana 300 s, de 285 a 315 s, em 719 consultas


## Passo 4 — o processo e o nome emprestado

O nome do processo não basta: **o atualizador legítimo do OneDrive roda na mesma estação**. O que separa
os dois é o caminho — o implante fica no perfil do usuário (`AppData`), onde não é preciso ser administrador
para escrever (\texttt{T1036.005 Masquerading}).

In [5]:
edr = eventos[eventos["event.dataset"] == "edr"]
na_estacao = edr[(edr["host.name"] == estacao) & (edr["process.name"] == "OneDriveUpdater.exe")]
print(na_estacao["process.executable"].value_counts())

# O atualizador legítimo mora em Arquivos de Programas; o implante, no perfil do usuário.
implante = na_estacao[na_estacao["process.executable"].str.contains("AppData")].iloc[0]
executavel = implante["process.executable"]
pai = implante["process.parent.name"]
print(executavel, "| pai:", pai)

process.executable
C:\Program Files\Microsoft OneDrive\OneDriveUpdater.exe           3
C:\Users\r.lemos\AppData\Roaming\Microsoft\OneDriveUpdater.exe    1
Name: count, dtype: int64
C:\Users\r.lemos\AppData\Roaming\Microsoft\OneDriveUpdater.exe | pai: WINWORD.EXE


## Passo 5 — a exfiltração que só aparece somando

O maior fluxo isolado é o backup interno. A hipótese pede **volume enviado para fora, somado por destino**.

In [6]:
firewall = eventos[eventos["event.dataset"] == "firewall"]
da_estacao = firewall[(firewall["host.name"] == estacao) &
                      (~firewall["destination.ip"].str.startswith("10."))]
somas = da_estacao.groupby("destination.ip")["source.bytes"].sum().sort_values(ascending=False)
destino = somas.index[0]
megabytes = int(somas.iloc[0] / 1_000_000)
print(destino, megabytes, "MB")
somas.head(5)

198.51.100.61 2292 MB


destination.ip
198.51.100.61    2.292934e+09
140.82.121.4     7.481520e+05
13.107.42.14     7.295990e+05
142.250.79.36    5.640100e+05
52.97.146.162    5.329600e+05
Name: source.bytes, dtype: float64

## Conferência contra o gabarito

As respostas do notebook precisam bater com as que o gerador recalcula por código sobre todos os eventos.

In [7]:
for chave, obtido in [("c2-dominio", dominio), ("c2-estacao", estacao),
                      ("c2-intervalo", intervalo), ("c2-executavel", executavel),
                      ("c2-pai", pai), ("c2-exfiltracao", destino), ("c2-volume", megabytes)]:
    print(conferir(chave, obtido, gabarito))

ok   c2-dominio: obtido 'telemetria-aurora-cdn.net', gabarito 'telemetria-aurora-cdn.net'
ok   c2-estacao: obtido 'WKS-ENG-117', gabarito 'WKS-ENG-117'
ok   c2-intervalo: obtido 300, gabarito '300'
ok   c2-executavel: obtido 'C:\\Users\\r.lemos\\AppData\\Roaming\\Microsoft\\OneDriveUpdater.exe', gabarito 'C:\\Users\\r.lemos\\AppData\\Roaming\\Microsoft\\OneDriveUpdater.exe'
ok   c2-pai: obtido 'WINWORD.EXE', gabarito 'WINWORD.EXE'
ok   c2-exfiltracao: obtido '198.51.100.61', gabarito '198.51.100.61'
ok   c2-volume: obtido 2292, gabarito '2292'


## Exercício

O implante deste caso consulta a cada 300 segundos exatos. Proponha uma medida de regularidade que continue funcionando se o intervalo variar de 240 a 360 segundos, e teste-a sobre os dados.